# Prise en main de PyTorch

**Objectifs de la séance.**
- Manipuler les tenseurs PyTorch.
- Comprendre le graphe de calcul et l'autodifférenciation (`requires_grad`, `.backward()`, `.grad`).
- Faire tourner une descente de gradient "à la main" en PyTorch, sur une fonction jouet puis sur une régression linéaire.

In [ ]:

import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)


## Partie 1 — Tenseurs

Un tenseur PyTorch ressemble beaucoup à un tableau `numpy` : il a une forme (`shape`), un type (`dtype`), et supporte les opérations usuelles (`+`, `*`, produit matriciel, etc.). La différence essentielle apparaîtra en partie 3 : un tenseur peut « se souvenir » des opérations dont il est issu, ce qui permettra le calcul automatique de gradients.

In [ ]:

# Quelques façons de créer des tenseurs
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.zeros(2, 3)
c = torch.ones(3, 2)
d = torch.rand(2, 2)          # tirage uniforme dans [0, 1)
e = torch.arange(0, 10, 2)

for name, t in [("a", a), ("b", b), ("c", c), ("d", d), ("e", e)]:
    print(f"{name}: shape={tuple(t.shape)}, dtype={t.dtype}")
    print(t)
    print()


**Question 1.1.** Le *broadcasting* permet d'effectuer des opérations entre tenseurs de formes différentes, à condition qu'elles soient « compatibles » (comme en `numpy`). Avant d'exécuter la cellule ci-dessous, essayez de prédire la forme du résultat de `x + y` pour chacune des trois paires proposées.

In [ ]:

x1, y1 = torch.rand(3, 1), torch.rand(1, 4)
x2, y2 = torch.rand(5, 3), torch.rand(3)
x3, y3 = torch.rand(2, 1, 4), torch.rand(3, 1)

for x, y in [(x1, y1), (x2, y2), (x3, y3)]:
    z = x + y
    print(f"{tuple(x.shape)} + {tuple(y.shape)} -> {tuple(z.shape)}")


**Question 1.2.** Créez un tenseur `m` de forme `(4, 5)` rempli de valeurs tirées selon une loi normale centrée réduite (indice : `torch.randn`), puis un vecteur `v` de forme `(5,)` rempli de 1. Calculez `m + v` et vérifiez que la forme du résultat est bien `(4, 5)`.


## Partie 2 — Différentiation manuelle en Python

Avant de nous appuyer sur PyTorch et l'autodifférenciation, voyons comment on ferait de la différentiation manuelle en Python pur.

On prend un exemple très simple en 1D : soit $x$ un scalaire et $y$ défini par

$$y = (x - 0.5)^2$$

L'objectif est de trouver la valeur de $x$ qui minimise $y$.

**Question 2.1.** Définissez une fonction `f` qui prend `x` en entrée et renvoie $y = (x-0.5)^2$.


Pour minimiser $y$, on va utiliser une descente de gradient : on met à jour $x$ itérativement en le déplaçant dans la direction opposée au gradient $\frac{\partial y}{\partial x}$. Il nous faut donc être capable de calculer $\frac{\partial y}{\partial x}$ — et comme on n'utilise pas encore l'autodifférenciation, il faut fournir la formule explicite de cette dérivée.

**Question 2.2.** Définissez une fonction `grad_f` qui prend `x` en entrée et renvoie $\frac{\partial y}{\partial x}$.


La règle de mise à jour de la descente de gradient s'écrit :

$$x \leftarrow x - \eta \frac{\partial y}{\partial x}$$

où $\eta$ (`eta`) est le pas d'apprentissage (*learning rate*).

**Question 2.3.** Choisissez une valeur de départ pour `x` et un pas `eta`, puis appliquez 30 itérations de descente de gradient (en utilisant `grad_f`). Affichez `x` à chaque itération.


**Question 2.4.** La valeur de `x` obtenue est-elle proche de celle attendue (le minimiseur de $y=f(x)$) ?


_Votre réponse ici._


## Partie 3 — Calcul automatique de gradients avec `autograd`

PyTorch ressemble beaucoup à `numpy`, à un détail près : on peut à tout moment demander le calcul automatique de gradients. Pour cela, il faut :

1. créer les tenseurs concernés avec `requires_grad=True` ;
2. calculer la quantité dont on veut le gradient (ex. `y = f(x)`) ;
3. appeler `y.backward()` : PyTorch calcule alors $\frac{\partial y}{\partial b}$ pour **tout** tenseur `b` ayant contribué au calcul de `y` (à condition que `b.requires_grad` soit `True`) ;
4. lire le résultat dans `b.grad`.

**Question 3.1.** Complétez le code ci-dessous pour calculer $y = f(x)$ avec $x = 0.125$, puis affichez `x.grad` *avant* d'avoir appelé `.backward()` : que vaut-il ?

In [ ]:

def f(x):
    return (x - 0.5) ** 2

x = torch.tensor(0.125, requires_grad=True)

# TODO : calculer y

print(x.grad)


**Question 3.2.** Déclenchez maintenant le calcul du gradient $\frac{\partial y}{\partial x}$ (via `.backward()`) et affichez-le. Vérifiez qu'il correspond à ce que donnerait `grad_f(x)` de la partie 2.


**Question 3.3.** Que se passe-t-il si vous encapsulez le calcul de `y` dans un bloc `with torch.no_grad():` ? Exécutez le code ci-dessous et regardez l'attribut `y.requires_grad`.

In [ ]:

x = torch.tensor(0.125, requires_grad=True)

with torch.no_grad():
    y = f(x)

print("y.requires_grad :", y.requires_grad)


## Partie 4 — Descente de gradient en PyTorch

**Question 4.1.** Reprenez la descente de gradient de la partie 2, mais en PyTorch cette fois : vous n'avez plus besoin de `grad_f`, autograd s'en charge. Chaque itération doit :

1. calculer `y` à partir de la valeur courante de `x` ;
2. déclencher le calcul du gradient ;
3. mettre à jour `x` (cette étape doit être protégée par `with torch.no_grad():`, sinon la mise à jour elle-même serait enregistrée dans le graphe de calcul) ;
4. remettre `x.grad` à zéro pour l'itération suivante (sans quoi PyTorch **accumule** les gradients au lieu de les remplacer, ligne déjà fournie ci-dessous).

In [ ]:

x = torch.tensor(0.125, requires_grad=True)

eta = 0.1
n_iter = 30

for i in range(n_iter):
    print(i, x.item())
    # TODO : calculer y et déclencher le calcul du gradient

    with torch.no_grad():
        pass  # TODO : mettre à jour x

    x.grad.zero_()  # ne pas modifier : remise à zéro du gradient


## Partie 5 — Pour aller plus loin : une régression linéaire

On termine cette séance en démarrant un exercice qui sera repris et terminé en séance 2 : ajuster un modèle de régression linéaire univarié $\hat y = wX + b$ à un jeu de données synthétique, par descente de gradient.

Le code ci-dessous génère et affiche le jeu de données (ne le modifiez pas : vous le retrouverez à l'identique en séance 2).

In [ ]:

X = torch.rand(100, 1)
# w* = -3, b* = 1.5
y = -3.0 * X + 1.5 + 0.4 * torch.randn(X.size())

plt.scatter(X.numpy(), y.numpy())
plt.xlabel("X"); plt.ylabel("y")


**Question 5.1.** D'après la façon dont le jeu de données a été généré, quelles devraient être les valeurs idéales de $w$ et $b$ ?


_Votre réponse ici._


**Question 5.2.** Implémentez une fonction `mse(X, y, w, b)` qui calcule l'erreur quadratique moyenne du modèle linéaire $\hat y = wX + b$ sur le jeu de données $(X, y)$.


**Question 5.3.** Initialisez `w` et `b` (tenseurs scalaires, `requires_grad=True`), puis écrivez une boucle de descente de gradient : à chaque itération, calculez la perte `mse`, déclenchez le calcul des gradients, mettez à jour `w` et `b` avec un pas `eta = 0.1`, puis remettez les gradients à zéro. Faites tourner une trentaine d'itérations en affichant la perte toutes les 5 itérations pour vérifier qu'elle diminue.

In [ ]:

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

eta = 0.1
n_iter = 30

for i in range(n_iter):
    # TODO : calculer la perte

    # TODO : déclencher le calcul des gradients

    with torch.no_grad():
        pass  # TODO : mettre à jour w et b

    # TODO : remettre w.grad et b.grad à zéro

    if i % 5 == 0:
        pass  # TODO : afficher la perte courante


## Bilan

Vous savez maintenant créer des tenseurs, calculer des gradients automatiquement, et coder une boucle de descente de gradient en PyTorch.